# Tool-Discovery Agent for Bioinformatics

Automatically discover bioinformatics tools from PubMed papers, extract GitHub repositories,
and convert them into standardized MCP (Model Context Protocol) tool interfaces.

**Pipeline:** PubMed search → Paper full-text → GitHub link extraction → Standardization → Cleaning → MCP Registry

In [ ]:
!pip install -q requests beautifulsoup4 pyyaml pandas tabulate python-dateutil

In [ ]:
import json, re, os, time, yaml
import requests
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd

WORK_DIR = "/content/tool_discovery"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")

## Step 1: Search PubMed & Extract GitHub Links

In [ ]:
def search_papers(query, max_results=10):
    """Search PubMed and return paper list."""
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {"db": "pubmed", "term": query, "retmax": max_results, "retmode": "json"}
    resp = requests.get(search_url, params=params, timeout=30)
    ids = resp.json().get("esearchresult", {}).get("idlist", [])
    if not ids:
        return []

    summary_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
    resp = requests.get(summary_url, params={"db": "pubmed", "id": ",".join(ids), "retmode": "json"}, timeout=30)
    data = resp.json()

    papers = []
    for uid in ids:
        doc = data.get("result", {}).get(uid, {})
        doi = ""
        for aid in doc.get("articleids", []):
            if aid.get("idtype") == "doi":
                doi = aid.get("value")
                break
        papers.append({
            "pmid": uid, "title": doc.get("title", ""),
            "abstract": doc.get("abstract", ""), "doi": doi,
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{uid}/"
        })
    return papers

def fetch_html_from_doi(doi):
    """Fetch paper HTML from DOI via Jina Reader (no Playwright needed)."""
    if not doi:
        return None
    paper_url = f"https://doi.org/{doi}"
    headers = {"User-Agent": "Mozilla/5.0 (compatible; ToolDiscovery/1.0)"}

    # Try Jina Reader first (works for most publishers)
    try:
        jina_url = f"https://r.jina.ai/{paper_url}"
        resp = requests.get(jina_url, headers=headers, timeout=30)
        if resp.status_code == 200 and len(resp.text) > 500:
            print(f"    Jina Reader: {len(resp.text)} chars")
            return resp.text
    except Exception as e:
        print(f"    Jina failed: {e}")

    # Fallback: follow redirects and parse HTML directly
    try:
        session = requests.Session()
        session.headers.update(headers)
        resp = session.get(paper_url, timeout=30, allow_redirects=True)
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        main = soup.find("article") or soup.find("main") or soup.find("body")
        if main:
            text = main.get_text(separator="\n", strip=True)
            if len(text) > 200:
                print(f"    Direct parse: {len(text)} chars")
                return text
    except Exception as e:
        print(f"    Direct parse failed: {e}")

    # Last resort: use abstract
    return None

def extract_github_links(text):
    """Extract bioinformatics-related GitHub links from text."""
    if not text:
        return []
    raw = list(set(re.findall(r'(https?://github\.com/[^\s\)<>]+)', text, re.IGNORECASE)))

    bio_kw = [
        "protein", "peptide", "enzyme", "fold", "structure", "design",
        "genome", "sequence", "alignment", "blast", "bio", "bioinfo",
        "tool", "server", "pipeline", "workflow", "package", "library",
        "docking", "simulation", "molecular", "pdb", "cif", "fasta",
        "sam", "bam", "vcf", "msa", "phylo", "variant", "rnaseq",
    ]
    irrelevant = ["normalize.css", "jquery", "bootstrap", "react", "angular", "vue", "d3", "three.js"]

    result = []
    for link in raw:
        low = link.lower()
        if any(kw in low for kw in bio_kw) and not any(ir in low for ir in irrelevant):
            result.append(link)
    return result

print("Functions defined.")

In [ ]:
# === Run Discovery ===
QUERY = "bioinformatics protein structure prediction tools"
MAX_PAPERS = 8

print(f"Searching PubMed: '{QUERY}'")
papers = search_papers(QUERY, max_results=MAX_PAPERS)
print(f"Found {len(papers)} papers\n")

discoveries = []
for i, paper in enumerate(papers, 1):
    print(f"[{i}/{len(papers)}] {paper['title'][:70]}...")
    html = fetch_html_from_doi(paper["doi"]) if paper.get("doi") else None
    if not html:
        html = paper.get("abstract", "")
    if not html or len(html) < 50:
        print("  No content, skipping.")
        continue

    links = extract_github_links(html)
    if links:
        print(f"  Found {len(links)} GitHub links: {links[:3]}")
        discoveries.append({
            "title": paper["title"][:100], "doi": paper["doi"],
            "github_links": links, "url": paper["url"]
        })
    else:
        print("  No GitHub links found.")
    time.sleep(0.5)

with open("github_from_html.json", "w", encoding="utf-8") as f:
    json.dump(discoveries, f, ensure_ascii=False, indent=2)

print(f"\nDiscovered {len(discoveries)} papers with GitHub tools.")

## Step 2: Standardize Tool Format

In [ ]:
def get_github_repo_info(repo_url):
    if not repo_url or "github.com" not in repo_url:
        return {"stars": 0, "description": "", "language": ""}
    parts = repo_url.rstrip("/").split("/")
    if len(parts) < 5:
        return {"stars": 0, "description": "", "language": ""}
    owner_repo = parts[3] + "/" + parts[4]
    try:
        resp = requests.get(f"https://api.github.com/repos/{owner_repo}", timeout=10)
        if resp.status_code == 200:
            d = resp.json()
            return {
                "stars": d.get("stargazers_count", 0),
                "description": d.get("description", ""),
                "language": d.get("language", ""),
                "updated_at": d.get("updated_at", ""),
                "forks": d.get("forks_count", 0),
            }
    except Exception:
        pass
    return {"stars": 0, "description": "", "language": ""}

def guess_tool_type(name, desc=""):
    text = (name + " " + desc).lower()
    if any(k in text for k in ["server", "web", "api"]): return "web_server"
    if any(k in text for k in ["database", "db"]): return "database"
    if any(k in text for k in ["pipeline", "workflow"]): return "pipeline"
    if any(k in text for k in ["library", "lib", "sdk"]): return "library"
    return "software"

def calculate_quality_score(gh_info, paper_count=1):
    score = min(gh_info.get("stars", 0) / 50 * 30, 30)
    if gh_info.get("description"): score += 10
    if gh_info.get("language"): score += 10
    updated = gh_info.get("updated_at", "")
    if updated:
        try:
            from dateutil.parser import parse
            days = (datetime.now() - parse(updated)).days
            if days < 30: score += 20
            elif days < 90: score += 10
        except Exception:
            pass
    score += min(paper_count * 5, 20)
    return min(score, 100)

def generate_tags(name, title, tool_type):
    tags = set()
    for w in re.sub(r'[^a-zA-Z ]', ' ', name).split():
        if len(w) > 2: tags.add(w.lower())
    bio_kw = ["protein", "design", "fold", "structure", "genome", "sequence",
              "bio", "enzyme", "docking", "simulation", "molecular"]
    for w in re.sub(r'[^a-zA-Z ]', ' ', title).split():
        if w.lower() in bio_kw: tags.add(w.lower())
    tags.add(tool_type)
    return list(tags)[:10]

def convert_to_standard(raw_results):
    tool_map = {}
    for item in raw_results:
        links = item.get("github_links", [])
        if not links: continue
        url = links[0]
        if url not in tool_map:
            tool_map[url] = {"github_url": url, "titles": [], "dois": [], "gh_info": get_github_repo_info(url)}
        tool_map[url]["titles"].append(item.get("title", ""))
        if item.get("doi"): tool_map[url]["dois"].append(item["doi"])
        time.sleep(0.3)  # rate limit GitHub API

    tools = []
    for url, data in tool_map.items():
        gh = data["gh_info"]
        name = url.split("/")[-1] or "unknown"
        desc = gh.get("description", "") or (data["titles"][0][:200] if data["titles"] else "")
        ttype = guess_tool_type(name, desc)
        tools.append({
            "name": name, "description": desc,
            "source": {"github": url, "paper_doi": data["dois"][0] if data["dois"] else "",
                       "paper_title": data["titles"][0] if data["titles"] else ""},
            "type": ttype,
            "interface": {"cli": True, "api": False, "web": ttype == "web_server"},
            "tags": generate_tags(name, data["titles"][0] if data["titles"] else "", ttype),
            "quality_score": calculate_quality_score(gh, len(data["titles"])),
            "github_metadata": {"stars": gh.get("stars", 0), "language": gh.get("language", ""),
                               "forks": gh.get("forks", 0)},
            "discovered_at": datetime.now().isoformat(),
        })
    tools.sort(key=lambda x: x["quality_score"], reverse=True)
    return tools

with open("github_from_html.json", "r", encoding="utf-8") as f:
    raw = json.load(f)
print(f"Loaded {len(raw)} raw discoveries")

standardized = convert_to_standard(raw)
with open("tool_library.json", "w", encoding="utf-8") as f:
    json.dump(standardized, f, ensure_ascii=False, indent=2)
print(f"Standardized: {len(standardized)} tools")

## Step 3: Clean & Filter

In [ ]:
irrelevant_patterns = [r'normalize\.css', r'jquery', r'bootstrap', r'react', r'angular',
                       r'vue\.js', r'd3\.js', r'three\.js']

filtered = []
for tool in standardized:
    combined = (tool.get("name", "") + " " + tool.get("description", "")).lower()
    if any(re.search(p, combined) for p in irrelevant_patterns):
        continue
    desc = tool.get("description", "").lower()
    if any(k in desc for k in ["database", "ontology"]): tool["type"] = "database"
    elif any(k in desc for k in ["pipeline", "workflow"]): tool["type"] = "pipeline"
    elif any(k in desc for k in ["server", "web"]): tool["type"] = "web_server"
    elif any(k in desc for k in ["library", "package"]): tool["type"] = "library"
    for kw in ["protein", "dna", "rna", "genome", "sequence", "structure",
               "binding", "folding", "design", "evolution", "antibody"]:
        if kw in desc and kw not in tool["tags"]:
            tool["tags"].append(kw)
    filtered.append(tool)

filtered.sort(key=lambda x: x.get("quality_score", 0), reverse=True)
with open("tool_library_clean.json", "w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=2)

print(f"After cleaning: {len(filtered)} tools (removed {len(standardized) - len(filtered)})")
df = pd.DataFrame([{"Name": t["name"], "Type": t["type"], "Stars": t["github_metadata"]["stars"],
                    "Score": t["quality_score"], "Language": t["github_metadata"]["language"]}
                   for t in filtered])
df

## Step 4: Convert to MCP Registry Format

In [ ]:
def tool_to_registry_entry(tool):
    name = tool.get("name", "unknown")
    clean_name = re.sub(r'[^a-zA-Z0-9_.-]', '_', name).strip('_').lower()
    gh_url = tool.get("source", {}).get("github", "")
    stars = tool.get("github_metadata", {}).get("stars", 0)
    lang = tool.get("github_metadata", {}).get("language", "")
    lang_flag = f", {lang}" if lang else ""
    return {
        "name": clean_name, "type": "cli",
        "command": f"{clean_name} {{{{input_file}}}}",
        "description": f"[Auto-discovered] {tool.get('description', '')} (\u2b50{stars}{lang_flag})",
        "output_control": {"intercept_large_output": True, "max_preview_lines": 50},
        "inputs": {"input_file": {"type": "string", "description": "Input file path."}},
        "_discovery_metadata": {
            "github": gh_url, "stars": stars, "language": lang,
            "tags": tool.get("tags", []),
            "quality_score": tool.get("quality_score", 0),
            "paper_doi": tool.get("source", {}).get("paper_doi", ""),
            "paper_title": tool.get("source", {}).get("paper_title", ""),
        },
    }

registry = {"tools": [tool_to_registry_entry(t) for t in filtered]}
with open("discovered_registry.yaml", "w", encoding="utf-8") as f:
    yaml.dump(registry, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f"Generated MCP registry with {len(registry['tools'])} tools → discovered_registry.yaml")
for t in registry["tools"][:5]:
    print(f"  - {t['name']}: {t['description'][:80]}")

## Step 5: Summary & Results

In [ ]:
print("=" * 60)
print("TOOL-DISCOVERY PIPELINE COMPLETE")
print("=" * 60)
print(f"\nPapers searched:       {MAX_PAPERS}")
print(f"Papers with tools:     {len(discoveries)}")
print(f"Tools discovered:      {len(standardized)}")
print(f"Tools after cleaning:  {len(filtered)}")
print(f"\nFiles generated:")
for f in ["github_from_html.json", "tool_library.json", "tool_library_clean.json", "discovered_registry.yaml"]:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f"  {f} ({size:,} bytes)")

print(f"\nTop tools by quality score:")
for t in filtered[:10]:
    print(f"  [{t['quality_score']:.0f}] {t['name']}: {t['description'][:60]}")

## (Optional) View Discovered Registry YAML

In [ ]:
with open("discovered_registry.yaml", "r", encoding="utf-8") as f:
    print(f.read())

## (Optional) Download Results

Run the cell below to download all result files as a zip.

In [ ]:
import zipfile
from google.colab import files

zip_path = "/content/tool_discovery_results.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for fname in ["github_from_html.json", "tool_library.json",
                  "tool_library_clean.json", "discovered_registry.yaml"]:
        if os.path.exists(fname):
            zf.write(fname)

files.download(zip_path)
print("Downloaded!")